# Resumen Observacional: Beta=0 vs Beta óptima

Compara, para los 8 experimentos "Observacional" (4 ruidos aditivos + 4 multiplicativos), el modelo base
(`Beta=0`, sin término HSIC) frente al modelo con la `Beta` óptima elegida automáticamente según un
criterio multi-métrica (`MAE Z`, `HSIC(Z,X)`, `HSIC(Z,Y)`, `RF Acc`), y comprueba con un test de
Wilcoxon pareado si las mejoras son significativas.

Todo el análisis se repite para dos tamaños muestrales, `N=50` y `N=100`, para ver si las conclusiones
son estables al cambiar N.


In [65]:
import pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"

ALPHA = 0.05
METRICAS = ["MAE Z", "HSIC(Z,X)", "HSIC(Z,Y)", "RF Acc"]  # menor es mejor en las 4


## Definición de los 8 experimentos

In [66]:
EXPERIMENTOS = {
    "Aditivo Gaussiano": {
        "path": NOTEBOOKS_DIR / "Experimento1/Observacional/tablas/aditivo_gausian.csv",
        "incompleto": False,
    },
    "Aditivo Exponencial": {
        "path": NOTEBOOKS_DIR / "Experimento1/Observacional/tablas/aditivo_exponential.csv",
        "incompleto": False,
    },
    "Aditivo Gamma": {
        "path": NOTEBOOKS_DIR / "Experimento1/Observacional/tablas/aditivo_gamma.csv",
        "incompleto": False,
    },
    "Aditivo Uniforme": {
        "path": NOTEBOOKS_DIR / "Experimento1/Observacional/tablas/aditivo_uniform.csv",
        "incompleto": False,
    },
    "Multiplicativo Gaussiano": {
        "path": NOTEBOOKS_DIR / "Experimento2/Observacional/tablas/multiplicativo_gausian.csv",
        "incompleto": False,
    },
    "Multiplicativo Exponencial": {
        "path": NOTEBOOKS_DIR / "Experimento2/Observacional/tablas/multiplicativo_exponencial.csv",
        "incompleto": True,  # CSV fuente a medias: notebook origen sigue en ejecucion
    },
    "Multiplicativo Gamma": {
        "path": NOTEBOOKS_DIR / "Experimento2/Observacional/tablas/multiplicativo_gamma.csv",
        "incompleto": False,
    },
    "Multiplicativo Uniforme": {
        "path": NOTEBOOKS_DIR / "Experimento2/Observacional/tablas/multiplicativo_uniforme.csv",
        "incompleto": False,
    },
}

for nombre, info in EXPERIMENTOS.items():
    assert info["path"].exists(), f"No existe: {info['path']}"


## Función de selección de la Beta óptima

In [67]:
def beta_optima(df: pd.DataFrame, n_filter: int, metrics=METRICAS):
    """Selecciona la Beta óptima (Beta != 0) de un experimento según un criterio multi-métrica.

    Beta=0 se usa solo como referencia (baseline) y nunca puede ser el resultado.

    1) Filtra N == n_filter, promedia cada métrica por Beta (sobre las seeds disponibles) y
       descarta Beta=0 del conjunto de candidatas.
    2) Para cada métrica, halla la Beta > 0 que minimiza su media -> betas candidatas.
    3) Para cada beta candidata, cuenta cuántas métricas mejoran/empeoran frente a Beta=0 y
       calcula la mejora relativa (%) de cada métrica frente al baseline.
    4) Se ordenan las candidatas por, en este orden de prioridad:
         a) menos métricas empeoradas (una candidata que no empeora ninguna gana siempre),
         b) más métricas mejoradas,
         c) mayor mejora relativa total (suma de mejoras relativas),
         d) ÚLTIMO desempate, solo si las anteriores empatan exactamente: mayor mejora
            relativa en una única métrica.
       Se elige la primera de ese orden.

    Devuelve (baseline: Series, beta_opt: float, valores_opt: Series, candidatas: DataFrame,
    criterio: str).
    """
    df_n = df[df["N"] == n_filter]
    media_por_beta = df_n.groupby("Beta")[list(metrics)].mean()

    baseline = media_por_beta.loc[0.0]
    media_no_cero = media_por_beta.drop(index=0.0)

    betas_candidatas = sorted(set(media_no_cero[m].idxmin() for m in metrics))

    filas_candidatas = []
    for beta in betas_candidatas:
        valores = media_no_cero.loc[beta]
        mejora_relativa = (baseline - valores) / baseline
        filas_candidatas.append({
            "Beta": beta,
            "n_mejoradas": int((valores < baseline).sum()),
            "n_empeoradas": int((valores > baseline).sum()),
            "mejora_relativa_total": float(mejora_relativa.sum()),
            "mejora_relativa_max": float(mejora_relativa.max()),
        })

    candidatas = pd.DataFrame(filas_candidatas).sort_values(
        by=["n_empeoradas", "n_mejoradas", "mejora_relativa_total", "mejora_relativa_max"],
        ascending=[True, False, False, False],
    ).reset_index(drop=True)

    ganadora = candidatas.iloc[0]
    beta_opt = float(ganadora["Beta"])
    valores_opt = media_por_beta.loc[beta_opt]

    if ganadora["n_empeoradas"] == 0:
        criterio = "domina al baseline (mejora >=1 metrica sin empeorar ninguna)"
    else:
        criterio = "mejor compromiso (ninguna beta evita empeorar alguna metrica)"

    return baseline, beta_opt, valores_opt, candidatas, criterio


## Función de test de Wilcoxon pareado

Para cada experimento y cada métrica, se comparan los valores por seed en `Beta=0` frente a
`Beta=beta_optima`, emparejados por `Seed`. Test unidireccional (`alternative='less'`): H0 = no hay
diferencia; H1 = el valor con la beta óptima es menor (mejor) que con beta=0.


In [68]:
def wilcoxon_experimento(df: pd.DataFrame, beta_opt: float, n_filter: int, metrics=METRICAS):
    """Wilcoxon signed-rank pareado (por Seed) entre Beta=0 y Beta=beta_opt, por métrica.

    H0: no hay diferencia. H1 (alternative='less'): el valor en beta_opt es menor (mejor) que en Beta=0.
    Devuelve (p_valores: dict metrica->p, n_pares: int).
    """
    df_n = df[df["N"] == n_filter]
    base = df_n[df_n["Beta"] == 0.0].set_index("Seed")[list(metrics)]
    opt = df_n[df_n["Beta"] == beta_opt].set_index("Seed")[list(metrics)]
    seeds_comunes = sorted(set(base.index) & set(opt.index))
    base = base.loc[seeds_comunes]
    opt = opt.loc[seeds_comunes]

    p_valores = {}
    for m in metrics:
        try:
            _, p = wilcoxon(opt[m].values, base[m].values, alternative="less")
        except ValueError:
            p = float("nan")
        p_valores[m] = p
    return p_valores, len(seeds_comunes)


## Cálculo (tabla resumen + p-valores) para un N dado

In [69]:
def analizar_n(n_filter: int):
    """Ejecuta beta_optima + wilcoxon_experimento para los 8 experimentos, a un N fijo.

    Devuelve (resumen: DataFrame, p_values: DataFrame, diagnostico: dict[str, DataFrame]).
    """
    filas_resumen = []
    filas_p = []
    diagnostico = {}

    for nombre, info in EXPERIMENTOS.items():
        df = pd.read_csv(info["path"])
        baseline, beta_opt, valores_opt, candidatas, criterio = beta_optima(df, n_filter=n_filter)
        diagnostico[nombre] = candidatas

        nota = "\u26a0 datos parciales" if info["incompleto"] else ""

        filas_resumen.append({
            "Experimento": nombre,
            "beta_optima": beta_opt,
            "MAE(Z) beta=0": baseline["MAE Z"],
            "MAE(Z) beta_optima": valores_opt["MAE Z"],
            "HSIC(Z,X) beta=0": baseline["HSIC(Z,X)"],
            "HSIC(Z,X) beta_optima": valores_opt["HSIC(Z,X)"],
            "HSIC(Z,Y) beta=0": baseline["HSIC(Z,Y)"],
            "HSIC(Z,Y) beta_optima": valores_opt["HSIC(Z,Y)"],
            "RF Acc beta=0": baseline["RF Acc"],
            "RF Acc beta_optima": valores_opt["RF Acc"],
            "Criterio": criterio,
            "Nota": nota,
        })

        p_valores, n_pares = wilcoxon_experimento(df, beta_opt, n_filter=n_filter)
        filas_p.append({"Experimento": nombre, **p_valores, "n_seeds_pareadas": n_pares})

    resumen = pd.DataFrame(filas_resumen)
    p_values = pd.DataFrame(filas_p).set_index("Experimento")
    return resumen, p_values, diagnostico


resumen_50, p_values_50, diagnostico_50 = analizar_n(50)
resumen_100, p_values_100, diagnostico_100 = analizar_n(100)


## Funciones de formato y resaltado en negrita

In [70]:
COLUMNA_A_METRICA = {
    "MAE(Z) beta_optima": "MAE Z",
    "HSIC(Z,X) beta_optima": "HSIC(Z,X)",
    "HSIC(Z,Y) beta_optima": "HSIC(Z,Y)",
    "RF Acc beta_optima": "RF Acc",
}


def formatear(resumen: pd.DataFrame):
    cols_num = [c for c in resumen.columns if c not in ("Experimento", "beta_optima", "Criterio", "Nota")]
    fmt = resumen.copy()
    fmt[cols_num] = fmt[cols_num].round(5)
    fmt["beta_optima"] = fmt["beta_optima"].round(2)
    return fmt


def tabla_con_negrita(resumen: pd.DataFrame, p_values: pd.DataFrame):
    """Tabla resumen con las celdas 'beta_optima' en negrita cuando su p-valor (Wilcoxon) < ALPHA.

    Solo aplica en la visualización del notebook (un CSV plano no admite negrita).
    """
    fmt = formatear(resumen)

    def resaltar(row):
        p_exp = p_values.loc[row["Experimento"]]
        estilos = []
        for col in row.index:
            metrica = COLUMNA_A_METRICA.get(col)
            if metrica is not None and pd.notna(p_exp[metrica]) and p_exp[metrica] < ALPHA:
                estilos.append("font-weight: bold")
            else:
                estilos.append("")
        return estilos

    return fmt.style.apply(resaltar, axis=1)


## N = 50

In [71]:
tabla_con_negrita(resumen_50, p_values_50)


,Experimento,beta_optima,MAE(Z) beta=0,MAE(Z) beta_optima,"HSIC(Z,X) beta=0","HSIC(Z,X) beta_optima","HSIC(Z,Y) beta=0","HSIC(Z,Y) beta_optima",RF Acc beta=0,RF Acc beta_optima,Criterio,Nota
0,Aditivo Gaussiano,0.100000,1.152110,1.142380,0.058760,0.054910,0.115160,0.113120,0.596250,0.597500,mejor compromiso (ninguna beta evita empeorar alguna metrica),
1,Aditivo Exponencial,0.300000,0.888200,0.885520,0.049410,0.044340,0.115100,0.111040,0.687500,0.688750,mejor compromiso (ninguna beta evita empeorar alguna metrica),
2,Aditivo Gamma,0.200000,1.160910,1.161000,0.170450,0.159650,0.218600,0.195790,0.607500,0.600000,mejor compromiso (ninguna beta evita empeorar alguna metrica),
3,Aditivo Uniforme,0.100000,0.989990,0.982340,0.045490,0.042800,0.058510,0.054890,0.605000,0.590000,domina al baseline (mejora >=1 metrica sin empeorar ninguna),
4,Multiplicativo Gaussiano,0.500000,1.016720,0.910800,0.151350,0.124190,0.073290,0.072610,0.622500,0.633750,mejor compromiso (ninguna beta evita empeorar alguna metrica),
5,Multiplicativo Exponencial,0.600000,0.719630,0.651790,0.229860,0.125690,0.059930,0.048730,0.752780,0.747220,domina al baseline (mejora >=1 metrica sin empeorar ninguna),⚠ datos parciales
6,Multiplicativo Gamma,0.600000,1.000840,0.757680,0.215460,0.157130,0.136270,0.097640,0.650000,0.618750,domina al baseline (mejora >=1 metrica sin empeorar ninguna),
7,Multiplicativo Uniforme,0.500000,0.762230,0.744000,0.110800,0.099060,0.085910,0.088210,0.616250,0.612500,mejor compromiso (ninguna beta evita empeorar alguna metrica),


In [72]:
p_values_50.round(4)


,MAE Z,"HSIC(Z,X)","HSIC(Z,Y)",RF Acc,n_seeds_pareadas
Experimento,,,,,
Aditivo Gaussiano,0.1671,0.1990,0.1990,0.5105,20
Aditivo Exponencial,0.2979,0.1153,0.1942,0.7367,20
Aditivo Gamma,0.2853,0.2814,0.0768,0.3109,20
Aditivo Uniforme,0.1012,0.0108,0.0884,0.0844,20
Multiplicativo Gaussiano,0.0664,0.0266,0.8159,0.7459,20
Multiplicativo Exponencial,0.0371,0.0371,0.3672,0.4297,9
Multiplicativo Gamma,0.0010,0.0379,0.0242,0.0298,20
Multiplicativo Uniforme,0.5796,0.1559,0.7510,0.3964,20


## N = 100

In [73]:
tabla_con_negrita(resumen_100, p_values_100)


,Experimento,beta_optima,MAE(Z) beta=0,MAE(Z) beta_optima,"HSIC(Z,X) beta=0","HSIC(Z,X) beta_optima","HSIC(Z,Y) beta=0","HSIC(Z,Y) beta_optima",RF Acc beta=0,RF Acc beta_optima,Criterio,Nota
0,Aditivo Gaussiano,0.100000,0.954930,0.956720,0.039320,0.039250,0.067360,0.068720,0.601250,0.595000,mejor compromiso (ninguna beta evita empeorar alguna metrica),
1,Aditivo Exponencial,0.300000,0.816590,0.771700,0.057420,0.046030,0.088230,0.061280,0.635000,0.648750,mejor compromiso (ninguna beta evita empeorar alguna metrica),
2,Aditivo Gamma,0.100000,0.835690,0.840820,0.105580,0.098960,0.099300,0.092160,0.617500,0.596250,mejor compromiso (ninguna beta evita empeorar alguna metrica),
3,Aditivo Uniforme,0.100000,0.956230,0.950780,0.032190,0.030810,0.047970,0.046330,0.585000,0.570000,domina al baseline (mejora >=1 metrica sin empeorar ninguna),
4,Multiplicativo Gaussiano,0.400000,0.797320,0.774590,0.081560,0.071580,0.072960,0.064280,0.598750,0.616250,mejor compromiso (ninguna beta evita empeorar alguna metrica),
5,Multiplicativo Exponencial,0.600000,0.682350,0.645060,0.198450,0.124810,0.063800,0.051150,0.716670,0.713890,domina al baseline (mejora >=1 metrica sin empeorar ninguna),⚠ datos parciales
6,Multiplicativo Gamma,0.400000,0.890710,0.709540,0.188120,0.120310,0.131720,0.108720,0.636250,0.625000,domina al baseline (mejora >=1 metrica sin empeorar ninguna),
7,Multiplicativo Uniforme,0.800000,0.714440,0.728750,0.090790,0.082400,0.097780,0.087350,0.606250,0.601250,mejor compromiso (ninguna beta evita empeorar alguna metrica),


In [74]:
p_values_100.round(4)


,MAE Z,"HSIC(Z,X)","HSIC(Z,Y)",RF Acc,n_seeds_pareadas
Experimento,,,,,
Aditivo Gaussiano,0.5938,0.2045,0.9285,0.3660,20
Aditivo Exponencial,0.0021,0.0016,0.0004,0.8487,20
Aditivo Gamma,0.2608,0.0348,0.0413,0.0459,20
Aditivo Uniforme,0.3271,0.4492,0.3371,0.1024,20
Multiplicativo Gaussiano,0.6629,0.2262,0.0045,0.8681,20
Multiplicativo Exponencial,0.0020,0.0137,0.0820,0.5000,9
Multiplicativo Gamma,0.0024,0.0042,0.0448,0.4217,20
Multiplicativo Uniforme,0.9958,0.1650,0.0077,0.5175,20


## Comparación N=50 vs N=100

Beta óptima elegida y número de métricas significativas (p < 0.05) en cada N, para ver de un vistazo
si las conclusiones cambian con el tamaño muestral.


In [75]:
comparacion = resumen_50[["Experimento", "beta_optima"]].rename(columns={"beta_optima": "beta_optima N=50"})
comparacion = comparacion.merge(
    resumen_100[["Experimento", "beta_optima"]].rename(columns={"beta_optima": "beta_optima N=100"}),
    on="Experimento",
)
comparacion["metricas_sig N=50"] = comparacion["Experimento"].map((p_values_50[METRICAS] < ALPHA).sum(axis=1))
comparacion["metricas_sig N=100"] = comparacion["Experimento"].map((p_values_100[METRICAS] < ALPHA).sum(axis=1))
comparacion


,Experimento,beta_optima N=50,beta_optima N=100,metricas_sig N=50,metricas_sig N=100
0,Aditivo Gaussiano,0.1,0.1,0,0
1,Aditivo Exponencial,0.3,0.3,0,3
2,Aditivo Gamma,0.2,0.1,0,3
3,Aditivo Uniforme,0.1,0.1,1,0
4,Multiplicativo Gaussiano,0.5,0.4,1,1
5,Multiplicativo Exponencial,0.6,0.6,2,2
6,Multiplicativo Gamma,0.6,0.4,4,3
7,Multiplicativo Uniforme,0.5,0.8,0,1


## Diagnóstico de candidatas (opcional)

Para inspeccionar, por experimento y N, qué betas fueron candidatas (óptimas para al menos una métrica)
y por qué se eligió la ganadora, descomenta y ejecuta:


In [76]:
# for nombre, tabla in diagnostico_50.items():
#     print(nombre, "(N=50)")
#     display(tabla)
# for nombre, tabla in diagnostico_100.items():
#     print(nombre, "(N=100)")
#     display(tabla)


## Guardar CSVs resumen

In [77]:
OUT_PATH_50 = NOTEBOOKS_DIR / "tablas" / "resumen_observacional_beta_n50.csv"
OUT_PATH_100 = NOTEBOOKS_DIR / "tablas" / "resumen_observacional_beta_n100.csv"

resumen_50.to_csv(OUT_PATH_50, index=False)
resumen_100.to_csv(OUT_PATH_100, index=False)

print(f"Guardado en: {OUT_PATH_50}")
print(f"Guardado en: {OUT_PATH_100}")


Guardado en: /Users/clau/Documents/TFM_NUEVO/CODIGO_COMPLETO/kacgm-hsic/kacgm_hsic/notebooks/tablas/resumen_observacional_beta_n50.csv
Guardado en: /Users/clau/Documents/TFM_NUEVO/CODIGO_COMPLETO/kacgm-hsic/kacgm_hsic/notebooks/tablas/resumen_observacional_beta_n100.csv
